# Ayundatia N°2: Problemas de optimización resueltos con Gurobi

## Optimización - IND3701-1

### Ayudantes: Antonia Púa & Marco Pérez

**Proposito de esta ayudantía:** Aplicar Gurobi a la resolución de problemas de optimización, comprendiendo la traducción de modelos matemáticos a código, las diferencias entre la versión gratuita y la licencia académica, y la interpretación de las soluciones obtenidas.

## **Gurobi versión gratuita vs Gurobi con licencia academica**

En este apartado se mostrará que la capacidad de la versión gratuita de Gurobi es limitada. En particular, al aumentar el tamaño de la instancia, llega un punto en que ya no es posible obtener una solución utilizando esa versión. Sin embargo, como estudiantes es posible solicitar una licencia académica, la cual permite resolver problemas de mayor tamaño.

Para ilustrar esto, se analizarán dos instancias del Problema 1 de la ayudantía anterior, que consistía en asignar el tiempo de estudio de un estudiante entre 4 exámenes, con el objetivo de maximizar su desempeño académico.

**Importante: cargar archivo c_ij.csv a "Archivos" acá en Colab**

### **P1. Modelo de asignación de tiempo de estudio en Gurobi**


In [ ]:
#instalar gurobi
!pip install gurobipy

In [ ]:
#importar librerias que necesitamos
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [ ]:
# =========================
# MODELO P1 EN GUROBI
# =========================

def modelo_asignacion(I,J,c, cursos):
  """ Parametros del modelo:
        - I
        - J
        - c
        - cursos
  """

  # Crear modelo
  m = gp.Model("P1_Asignacion_Tiempo_Estudio")

  # Variable binaria:
  # x[i,j] = 1 si al curso i se le asignan j bloques
  x = m.addVars(I, J, vtype=GRB.BINARY, name="x")

  # Función objetivo:
  # maximizar suma de notas
  m.setObjective(
      gp.quicksum(c[i, j] * x[i, j] for i in I for j in J),
      GRB.MAXIMIZE
  )

  # Restricción 1:
  # para cada curso se elige exactamente una cantidad de bloques
  for i in I:
      m.addConstr(
          gp.quicksum(x[i, j] for j in J) == 1,
          name=f"un_bloque_por_curso_{i}"
      )

  # Restricción 2:
  # no usar más de 5 bloques en total (20 horas / 4 = 5 bloques)
  m.addConstr(
      gp.quicksum(j * x[i, j] for i in I for j in J) <= 5,
      name="max_bloques"
  )

  # Restricción 3:
  # Termodinámica (i = 3) debe tener nota al menos 4
  m.addConstr(
      gp.quicksum(c[3, j] * x[3, j] for j in J) >= 4,
      name="min_nota_termodinamica"
  )

  # Restricción 4:
  # Minería de Datos (i = 4) debe tener nota al menos 3
  m.addConstr(
      gp.quicksum(c[4, j] * x[4, j] for j in J) >= 3,
      name="min_nota_mineria"
  )

  # Optimizar
  m.optimize()

  nombre_curso = dict(zip(cursos["i"], cursos["curso"]))

  # Mostrar resultados
  if m.status == GRB.OPTIMAL:
      print(f"Valor óptimo = {m.objVal:.2f}\n")

      total_bloques = 0
      total_notas = 0

      for i in I:
          for j in J:
              if x[i, j].X > 0.5:
                  nota = c[i, j]
                  total_bloques += j
                  total_notas += nota
                  print(f"Curso: {nombre_curso[i]:25s} | Bloques asignados: {j} | Nota: {nota}")

      print("\nResumen:")
      print(f"Bloques totales usados: {total_bloques}")
      print(f"Suma total de notas: {total_notas:.2f}")

  else:
      print("No se encontró solución óptima.")

In [ ]:
# Codificar los parametros que necesita el modelo

# Leer datos
df = pd.read_csv("c_ij.csv")

# Conjuntos
I = sorted(df["i"].unique())           # cursos
J = sorted(df["j_bloques"].unique())   # bloques posibles

# Parámetro c_ij
c = {(int(row.i), int(row.j_bloques)): float(row.c_ij) for _, row in df.iterrows()}

# Nombres de cursos (solo para mostrar resultados)
cursos = (
    df[["i", "curso"]]
    .drop_duplicates()
    .sort_values("i")
)

In [ ]:
#Ejecutar modelo con los parametros definidos anteriormente
modelo_asignacion(I,J,c,cursos)

### **P2. Modelo de asignación de tiempo para proyectos en Gurobi**

In [ ]:
def resolver_asignacion_proyectos(I, J, c, G1, G2, B, L1, L2, verbose=True, env = None):
    """
    I  : conjunto de proyectos
    J  : conjunto de bloques posibles por proyecto
    c  : diccionario con impacto esperado c[i,j]
    G1 : primer grupo de proyectos importantes
    G2 : segundo grupo de proyectos importantes
    B  : cantidad total disponible de bloques
    L1 : impacto mínimo exigido para G1
    L2 : impacto mínimo exigido para G2
    """

    m = gp.Model("Asignacion_Proyectos", env = env)

    if not verbose:
        m.Params.OutputFlag = 0

    # Variable binaria:
    # x[i,j] = 1 si al proyecto i se le asignan j bloques
    x = m.addVars(I, J, vtype=GRB.BINARY, name="x")

    # Función objetivo: maximizar impacto total esperado
    m.setObjective(
        gp.quicksum(c[i, j] * x[i, j] for i in I for j in J),
        GRB.MAXIMIZE
    )

    # 1) Cada proyecto recibe exactamente un nivel de dedicación (o solo una vez se decide esto)
    m.addConstrs(
        (gp.quicksum(x[i, j] for j in J) == 1 for i in I),
        name="asignacion_unica"
    )

    # 2) No exceder los bloques totales disponibles
    m.addConstr(
        gp.quicksum(j * x[i, j] for i in I for j in J) <= B,
        name="presupuesto_bloques"
    )

    # 3) Impacto mínimo para el grupo G1
    m.addConstr(
        gp.quicksum(c[i, j] * x[i, j] for i in G1 for j in J) >= L1,
        name="impacto_minimo_G1"
    )

    # 4) Impacto mínimo para el grupo G2
    m.addConstr(
        gp.quicksum(c[i, j] * x[i, j] for i in G2 for j in J) >= L2,
        name="impacto_minimo_G2"
    )

    m.optimize()

    return m, x

### **P3. Ejecución del modelo P2 con instancia ficticia de gran escala**

In [ ]:
#Creación de instancia empresarial ficticia

import numpy as np
import pandas as pd

# -----------------------------
# CONJUNTOS
# -----------------------------
n_proyectos = 260
I = list(range(1, n_proyectos + 1))   # proyectos 1,...,260
J = list(range(0, 9))                 # 0,1,2,...,8 bloques posibles

# Dos grupos importantes
G1 = list(range(1, 31))               # proyectos 1 al 30
G2 = list(range(31, 61))              # proyectos 31 al 60

# Presupuesto total de bloques
B = 300

# -----------------------------
# PARÁMETROS c_ij
# -----------------------------
# Generamos impactos crecientes con los bloques,
# pero con rendimientos marginales decrecientes.
rng = np.random.default_rng(42)

# "base[i]" representa la relevancia/retorno base del proyecto i
base = {i: int(rng.integers(12, 35)) for i in I}

c = {}
for i in I:
    for j in J:
        if j == 0:
            c[i, j] = 0.0
        else:
            c[i, j] = round(base[i] * (1 - np.exp(-0.45 * j)) + 0.6 * j, 2)

# -----------------------------
# NIVELES MÍNIMOS DE IMPACTO
# -----------------------------
# Elegimos L1 y L2 de forma que la instancia sea factible.
L1 = round(0.90 * sum(c[i, 2] for i in G1), 2)
L2 = round(0.90 * sum(c[i, 2] for i in G2), 2)

print("L1 =", L1)
print("L2 =", L2)

In [ ]:
m, x = resolver_asignacion_proyectos(I, J, c, G1, G2, B, L1, L2, verbose=True)

### **P4. Configurar entorno de licencia y ejecutar modelo**

In [ ]:
# Reemplaza estos valores por los datos de tu licencia WLS antes de ejecutar esta celda.
# No subas tus credenciales reales a GitHub.
%%writefile gurobi.env
WLSACCESSID=TU_WLSACCESSID
WLSSECRET=TU_WLSSECRET
LICENSEID=TU_LICENSEID


In [ ]:
# Crear entorno de Gurobi con licencia WLS
from gurobipy import *

env = Env(empty=True)
env.readParams("gurobi.env")
env.start()


In [ ]:
m, x = resolver_asignacion_proyectos(I, J, c, G1, G2, B, L1, L2, verbose=True, env= env)

In [ ]:
#Recuperar e imprimir solución

if m.Status == GRB.OPTIMAL:
    print(f"\nValor óptimo: {m.ObjVal:.2f}\n")

    asignaciones = []
    for i in I:
        for j in J:
            if x[i, j].X > 0.5:
                asignaciones.append([i, j, c[i, j]])

    df_sol = pd.DataFrame(asignaciones, columns=["Proyecto", "Bloques_asignados", "Impacto"])
    print(df_sol.head(20))

    print("\nBloques totales usados:", df_sol["Bloques_asignados"].sum())
    print("Impacto total:", round(df_sol["Impacto"].sum(), 2))

    impacto_G1 = df_sol[df_sol["Proyecto"].isin(G1)]["Impacto"].sum()
    impacto_G2 = df_sol[df_sol["Proyecto"].isin(G2)]["Impacto"].sum()

    print("Impacto total en G1:", round(impacto_G1, 2))
    print("Impacto total en G2:", round(impacto_G2, 2))

elif m.Status == GRB.INFEASIBLE:
    print("El modelo resultó infactible.")
else:
    print("Estado del modelo:", m.Status)

## **Problema de localización: formulación e implementación en Gurobi**

### **Preparación para Google Colab**

Si solo quieres ejecutar el problema de localización, corre desde esta celda en adelante. La instancia es pequeña, por lo que puede resolverse con la instalación normal de `gurobipy` en Colab; si quieres usar tu licencia académica WLS, primero ejecuta las celdas de configuración de licencia de la sección anterior.

In [ ]:
# Instalar Gurobi en Colab si no está instalado
try:
    import gurobipy as gp
    from gurobipy import GRB
except ModuleNotFoundError:
    !pip install gurobipy
    import gurobipy as gp
    from gurobipy import GRB

# Librerías usadas en la instancia y visualización
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("Librerías cargadas correctamente.")

### **P1. Análisis del modelo matemático**

El problema busca decidir dónde abrir farmacias públicas y cómo asignar su capacidad para maximizar la demanda efectivamente atendida.

- **Conjuntos:** `I` son las manzanas con demanda, `J` son las ubicaciones candidatas para abrir farmacias y `M` son los tipos de medicamentos.
- **Parámetros:** `p[i,m]` es la demanda de medicamento `m` en la manzana `i`, `c[j]` es el costo de abrir en `j`, `B` es el presupuesto, `CAP[j]` es la capacidad de la farmacia `j`, `d[i,j]` es la distancia y `R` es el radio máximo de cobertura.
- **Variables:** `x[j]` indica si se abre la farmacia `j`, `y[m,j]` indica cuánto medicamento `m` se ofrece en `j`, y `z[i,m,j]` indica cuánto medicamento `m` consume la manzana `i` desde la farmacia `j`.
- **Objetivo:** maximizar la suma total de medicamentos atendidos.
- **Restricciones:** respetar presupuesto, activar capacidad solo si la farmacia se abre, no atender más demanda que la existente y no entregar más medicamentos que los ofrecidos por cada farmacia.

En la implementación se crean variables `z[i,m,j]` solo cuando la farmacia `j` está dentro del radio de cobertura de la manzana `i`. Esto representa la condición `d[i,j] < R` directamente en el código y evita asignaciones fuera de cobertura.

In [ ]:
# =========================
# MODELO DE LOCALIZACIÓN EN GUROBI
# =========================

def resolver_localizacion(I, J, M, p, c, B, CAP, d, R, verbose=True, env=None):
    """
    I   : conjunto de manzanas con demanda
    J   : conjunto de ubicaciones candidatas para instalar farmacias
    M   : conjunto de tipos de medicamentos
    p   : demanda promedio p[i,m]
    c   : costo de instalar farmacia en j
    B   : presupuesto total disponible
    CAP : capacidad total de la farmacia j
    d   : distancia d[i,j]
    R   : radio máximo de cobertura
    """

    # Ubicaciones que cubren a cada manzana y manzanas cubiertas por cada ubicación
    N = {i: [j for j in J if d[i, j] < R] for i in I}
    N_hat = {j: [i for i in I if d[i, j] < R] for j in J}

    # Solo se permiten flujos dentro del radio de cobertura
    arcos_cobertura = [(i, m, j) for i in I for m in M for j in N[i]]

    if env is None:
        m = gp.Model("Localizacion_Farmacias")
    else:
        m = gp.Model("Localizacion_Farmacias", env=env)

    if not verbose:
        m.Params.OutputFlag = 0

    # x[j] = 1 si se instala una farmacia en j
    x = m.addVars(J, vtype=GRB.BINARY, name="x")

    # y[m,j] = cantidad del medicamento m ofrecida en la farmacia j
    y = m.addVars(M, J, lb=0, vtype=GRB.CONTINUOUS, name="y")

    # z[i,m,j] = cantidad del medicamento m consumida por la manzana i desde j
    z = m.addVars(arcos_cobertura, lb=0, vtype=GRB.CONTINUOUS, name="z")

    # (1) Maximizar demanda total atendida
    m.setObjective(
        gp.quicksum(z[i, med, j] for i, med, j in arcos_cobertura),
        GRB.MAXIMIZE
    )

    # (2) Presupuesto disponible para abrir farmacias
    m.addConstr(
        gp.quicksum(c[j] * x[j] for j in J) <= B,
        name="presupuesto"
    )

    # (3) Capacidad total de cada farmacia, activada solo si se abre
    m.addConstrs(
        (gp.quicksum(y[med, j] for med in M) <= CAP[j] * x[j] for j in J),
        name="capacidad"
    )

    # (4) La demanda atendida no puede superar la demanda existente
    m.addConstrs(
        (gp.quicksum(z[i, med, j] for j in N[i]) <= p[i, med]
         for i in I for med in M),
        name="demanda"
    )

    # (5) Cada farmacia no puede entregar más de lo que ofrece por medicamento
    m.addConstrs(
        (gp.quicksum(z[i, med, j] for i in N_hat[j]) <= y[med, j]
         for med in M for j in J),
        name="oferta"
    )

    m.optimize()

    return m, x, y, z, N, N_hat

### **P2. Implementación computacional con una instancia pequeña**

La función `resolver_localizacion` sigue el mismo orden del modelo matemático:

1. Construye los conjuntos de cobertura `N[i]` y `N_hat[j]` usando la condición `d[i,j] < R`.
2. Crea las variables `x`, `y` y `z`; en particular, `z` solo existe para pares manzana-farmacia dentro del radio de cobertura.
3. Define la función objetivo como la suma de toda la demanda atendida.
4. Agrega las restricciones de presupuesto, capacidad, demanda máxima por manzana y oferta máxima por farmacia.
5. Optimiza y retorna el modelo, las variables y los conjuntos de cobertura para poder analizar la solución.

La siguiente instancia ficticia permite ejecutar el modelo de forma directa. Las coordenadas se usan solo para calcular distancias y visualizar la solución.

In [ ]:
# -----------------------------
# DATOS DE EJEMPLO
# -----------------------------

# Manzanas con demanda
I = list(range(1, 9))

# Ubicaciones candidatas para instalar farmacias
J = [1, 3, 5, 7]

# Tipos de medicamentos
M = ["Analgésico", "Antibiótico", "Crónico"]

# Coordenadas ficticias de cada manzana, en kilómetros
coords = {
    1: (0.0, 0.0),
    2: (0.6, 0.2),
    3: (1.2, 0.1),
    4: (0.2, 0.9),
    5: (1.0, 0.8),
    6: (1.8, 0.4),
    7: (1.9, 1.1),
    8: (2.5, 0.9),
}

# Demanda promedio por manzana y medicamento
demanda = pd.DataFrame({
    "Manzana": I,
    "Analgésico":  [12, 10, 14,  8, 13,  9, 11,  7],
    "Antibiótico": [ 5,  6,  7,  4,  6,  5,  6,  4],
    "Crónico":     [18, 15, 16, 12, 17, 14, 15, 10],
})

p = {
    (int(row["Manzana"]), med): float(row[med])
    for _, row in demanda.iterrows()
    for med in M
}

# Costos de instalación, presupuesto y capacidades
c = {1: 45, 3: 50, 5: 48, 7: 42}
B = 95
CAP = {1: 45, 3: 50, 5: 55, 7: 48}

# Radio máximo de cobertura
R = 1.15

# Distancias euclidianas entre manzanas y ubicaciones candidatas
d = {
    (i, j): math.dist(coords[i], coords[j])
    for i in I
    for j in J
}

print("Demanda total:", sum(p[i, med] for i in I for med in M))
print("Presupuesto disponible:", B)
print("Ubicaciones candidatas:", J)

### **Visualización de la instancia antes de optimizar**

Antes de resolver el modelo, se muestra la instancia base: las manzanas con demanda y las ubicaciones candidatas donde se podría instalar una farmacia. En este gráfico todavía no hay decisión de apertura; solo se está representando la información de entrada del problema.

In [ ]:
# -----------------------------
# VISUALIZACIÓN DE LA INSTANCIA
# -----------------------------

fig, ax = plt.subplots(figsize=(7, 5))

# Todas las manzanas con demanda
ax.scatter(
    [coords[i][0] for i in I],
    [coords[i][1] for i in I],
    s=80,
    color="lightgray",
    edgecolor="black",
    label="Manzanas con demanda"
)

# Ubicaciones candidatas
ax.scatter(
    [coords[j][0] for j in J],
    [coords[j][1] for j in J],
    s=160,
    marker="s",
    facecolors="none",      # relleno transparente,
    edgecolor="tab:blue",
    linewidth=2,
    label="Ubicaciones candidatas"
)

# Radio de cobertura de cada candidata, solo como información de la instancia
for j in J:
    circulo = plt.Circle(coords[j], R, color="tab:blue", alpha=0.08, linestyle="--", fill=True)
    ax.add_patch(circulo)

for i in I:
    ax.text(coords[i][0] + 0.03, coords[i][1] + 0.03, str(i), fontsize=10)

ax.set_title("Instancia del problema de localización")
ax.set_xlabel("Coordenada x [km]")
ax.set_ylabel("Coordenada y [km]")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
plt.show()

In [ ]:
# Ejecutar el modelo de localización
# Si existe un entorno WLS creado antes con nombre "env", se usa automáticamente.
try:
    env_localizacion = env
except NameError:
    env_localizacion = None

m_loc, x_loc, y_loc, z_loc, N, N_hat = resolver_localizacion(
    I, J, M, p, c, B, CAP, d, R, verbose=True, env=env_localizacion
)


### **P3. Recuperación y análisis de resultados**

Las siguientes tablas muestran qué farmacias se abren, cuánta capacidad usan y cuánta demanda se atiende por manzana y medicamento.

In [ ]:
# -----------------------------
# RECUPERAR E IMPRIMIR SOLUCIÓN
# -----------------------------

if m_loc.Status == GRB.OPTIMAL:
    print(f"Valor óptimo: {m_loc.ObjVal:.2f}\n")

    farmacias = []
    for j in J:
        abierta = x_loc[j].X > 0.5
        ofrecido = sum(y_loc[med, j].X for med in M)
        entregado = sum(
            z_loc[i, med, j].X
            for i in N_hat[j]
            for med in M
            if (i, med, j) in z_loc
        )

        farmacias.append({
            "Ubicación": j,
            "Abierta": "Sí" if abierta else "No",
            "Costo": c[j],
            "Capacidad": CAP[j],
            "Capacidad usada": round(entregado, 2),
            "% uso capacidad": round(100 * entregado / CAP[j], 1),
        })

    df_farmacias = pd.DataFrame(farmacias)
    print("Farmacias candidatas:")
    display(df_farmacias)

    atencion = []
    for i in I:
        for med in M:
            atendido = sum(
                z_loc[i, med, j].X
                for j in N[i]
                if (i, med, j) in z_loc
            )
            atencion.append({
                "Manzana": i,
                "Medicamento": med,
                "Demanda": p[i, med],
                "Atendido": round(atendido, 2),
                "% atendido": round(100 * atendido / p[i, med], 1) if p[i, med] > 0 else 0,
            })

    df_atencion = pd.DataFrame(atencion)
    print("\nDemanda atendida por manzana y medicamento:")
    display(df_atencion)

    flujos = []
    for i, med, j in z_loc.keys():
        if z_loc[i, med, j].X > 1e-6:
            flujos.append({
                "Manzana": i,
                "Medicamento": med,
                "Farmacia": j,
                "Cantidad": round(z_loc[i, med, j].X, 2),
            })

    df_flujos = pd.DataFrame(flujos)
    print("\nFlujos positivos de atención:")
    display(df_flujos)

    print("Resumen:")
    print("Farmacias abiertas:", [j for j in J if x_loc[j].X > 0.5])
    print("Costo total:", sum(c[j] * x_loc[j].X for j in J))
    print("Demanda total atendida:", round(m_loc.ObjVal, 2))
    print("Demanda total existente:", sum(p[i, med] for i in I for med in M))

elif m_loc.Status == GRB.INFEASIBLE:
    print("El modelo resultó infactible.")
elif m_loc.Status == GRB.UNBOUNDED:
    print("El modelo resultó no acotado.")
else:
    print("Estado del modelo:", m_loc.Status)

In [ ]:
# -----------------------------
# VISUALIZACIÓN DE LA SOLUCIÓN
# -----------------------------

if m_loc.Status == GRB.OPTIMAL:
    fig, ax = plt.subplots(figsize=(7, 5))

    # Manzanas con demanda
    ax.scatter(
        [coords[i][0] for i in I],
        [coords[i][1] for i in I],
        s=80,
        color="lightgray",
        edgecolor="black",
        label="Manzanas"
    )

    # Ubicaciones candidatas cerradas
    cerradas = [j for j in J if x_loc[j].X <= 0.5]
    ax.scatter(
        [coords[j][0] for j in cerradas],
        [coords[j][1] for j in cerradas],
        s=140,
        marker="s",
        facecolors="none",
        edgecolor="tab:red",
        linewidth=2,
        label="Candidata no abierta"
    )

    # Farmacias abiertas y radios de cobertura
    abiertas = [j for j in J if x_loc[j].X > 0.5]
    ax.scatter(
        [coords[j][0] for j in abiertas],
        [coords[j][1] for j in abiertas],
        s=180,
        marker="s",
        facecolors="none",
        edgecolor="green",
        linewidth=2,
        label="Farmacia abierta"
    )

    for j in abiertas:
        circulo = plt.Circle(coords[j], R, color="tab:green", alpha=0.12)
        ax.add_patch(circulo)

    for i in I:
        ax.text(coords[i][0] + 0.03, coords[i][1] + 0.03, str(i), fontsize=10)

    ax.set_title("Solución del problema de localización")
    ax.set_xlabel("Coordenada x [km]")
    ax.set_ylabel("Coordenada y [km]")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.show()

La visualización permite identificar las ubicaciones seleccionadas, las manzanas dentro del radio de cobertura y las farmacias candidatas que no fueron abiertas. Las tablas complementan el gráfico indicando el uso de capacidad y la demanda realmente atendida, que corresponde al valor de la función objetivo.